In [1]:
import pandas as pd

train=pd.read_csv("engtamilTrain.csv")
train=train.drop(["Unnamed: 0"],axis=1)
english_sentences=train["en"]
tamil_sentence=train['ta']
english_sentences=english_sentences.head(1000)
tamil_sentences=tamil_sentence.head(1000)


In [2]:
english_sentences.tail(5)


995    A face that stays on in people's mind is impor...
996    According to a report distributed by the Pales...
997    In a fundraising letter to big contributors, B...
998    It is entirely appropriate that outraged prote...
999    Against him there exists nothing, and he knows...
Name: en, dtype: object

In [3]:
tamil_sentences.tail(5)


995    ஒரு நடிகருக்கோ, நடிகைக்கோ மனதில் பதியக்கூடிய ம...
996    ஒரு அகதி முகாமின் ஒரு தெருவில் 30 பேர் கொல்லப...
997    புஷ் பிரச்சார பொது வக்கீலான Tom Josejiak விடு...
998    ஆத்திரம் கொண்ட ஆர்ப்பாட்டக்காரர்கள் CNN, A.B....
999    அவருக்கு எதிராக ஒன்றும் இல்லை; அதை அவர் நன்கு ...
Name: ta, dtype: object

In [4]:
#Function to add SOS and EOS to the statement

def addSosEos(seriesSentence):
    # Define the <SOS> and <EOS> tokens
    sos_token = "<SOS>"
    eos_token = "<EOS>"

    # Add <SOS> and <EOS> tokens to each statement
    statements_with_tokens = [f"{sos_token} {statement} {eos_token}" for statement in seriesSentence]

    english_sent=[]
    # Add statements with tokens
    for statement in statements_with_tokens:
        english_sent.append(statement)
    return english_sent


In [5]:
english_sent_SE=addSosEos(english_sentences)


In [6]:
print(english_sentences[1:5])
print('\nAfter adding SOS and EOS tokens\n')
print(english_sent_SE[1:5])


1    Information has surfaced in recent years sugge...
2    And Azor begat Sadoc; and Sadoc begat Achim; a...
3    She says she knows what is going on, but can d...
4    And be it indeed that I have erred, my error r...
Name: en, dtype: object

After adding SOS and EOS tokens

['<SOS> Information has surfaced in recent years suggesting that Julius Rosenberg was involved in passing some form of intelligence to Soviet officials during the Second World War.\n <EOS>', '<SOS> And Azor begat Sadoc; and Sadoc begat Achim; and Achim begat Eliud;\n <EOS>', '<SOS> She says she knows what is going on, but can do nothing about it.\n <EOS>', '<SOS> And be it indeed that I have erred, my error remains with myself.\n <EOS>']


In [7]:
tamil_sent_SE=addSosEos(tamil_sentences)


In [8]:
print(tamil_sentences[1:5])
print('\nAfter adding SOS and EOS tokens\n')
print(tamil_sent_SE[1:5])


1    சமீபகாலத்தில் சில தகவல்கள் யூலியஸ் ரோசன்பேர்க...
2    ஆசோர் சாதோக்கைப் பெற்றான்; சாதோக்கு ஆகீமைப் பெ...
3    என்ன நடக்கிறது என்பது தமக்கு தெரியும் என்றும் ...
4    நான் தப்பிநடந்தது மெய்யானாலும், என் தப்பிதம் எ...
Name: ta, dtype: object

After adding SOS and EOS tokens

['<SOS> சமீபகாலத்தில் சில தகவல்கள் யூலியஸ் ரோசன்பேர்க் ஒரு வித உளவுச்செய்தியை சோவியத் அதிகாரிகளுக்கு இரண்டாம் உலகப்போரின்போது அனுப்பியதில் சம்பந்தப்பட்டு இருந்ததாக வெளிவந்துள்ளன.\n <EOS>', '<SOS> ஆசோர் சாதோக்கைப் பெற்றான்; சாதோக்கு ஆகீமைப் பெற்றான்; ஆகீம் எலியூதைப் பெற்றான்;\n <EOS>', '<SOS> என்ன நடக்கிறது என்பது தமக்கு தெரியும் என்றும் ஆனால், தம்மால் எதுவும் செய்யமுடியாது என்றும் கடிதம் எழுதியிருந்தார்.\n <EOS>', '<SOS> நான் தப்பிநடந்தது மெய்யானாலும், என் தப்பிதம் என்னோடேதான் இருக்கிறது\n <EOS>']


In [9]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [10]:
# Tokenize the English and Tamil sentences
english_tokenizer = Tokenizer(filters="")
english_tokenizer.fit_on_texts(english_sent_SE)
english_vocab_size = len(english_tokenizer.word_index) + 1
english_sequences = english_tokenizer.texts_to_sequences(english_sent_SE)


In [ ]:
# Print the word index of the English tokenizer
list(english_tokenizer.word_index.items())[:5]


[('the', 1), ('<sos>', 2), ('<eos>', 3), ('of', 4), ('and', 5)]

In [30]:
print('Vocabulary size of English words:', english_vocab_size)
print(english_sent_SE[1])
print(english_sequences[1], '\nLength of sequence:', len(english_sequences[1]))


Vocabulary size of English words: 6979
<SOS> Information has surfaced in recent years suggesting that Julius Rosenberg was involved in passing some form of intelligence to Soviet officials during the Second World War.
 <EOS>
[2, 762, 18, 2042, 7, 567, 143, 2043, 9, 2044, 2045, 15, 568, 7, 2046, 126, 1114, 4, 340, 6, 446, 447, 184, 1, 341, 105, 342, 3] 
Length of sequence: 28


In [31]:
tamil_tokenizer = Tokenizer(filters="")
tamil_tokenizer.fit_on_texts(tamil_sent_SE)
tamil_vocab_size = len(tamil_tokenizer.word_index) + 1
tamil_sequences = tamil_tokenizer.texts_to_sequences(tamil_sent_SE)


In [32]:
# Print the word index of the Tamil tokenizer
list(tamil_tokenizer.word_index.items())[:5]


[('<sos>', 1), ('<eos>', 2), ('ஒரு', 3), ('மற்றும்', 4), ('என்று', 5)]

In [33]:
print('Vocabulary size of English words:', english_vocab_size)
print(tamil_sent_SE[1])
print(tamil_sequences[1], '\nLength of sequence:', len(tamil_sequences[1]))


Vocabulary size of English words: 6979
<SOS> சமீபகாலத்தில் சில தகவல்கள் யூலியஸ் ரோசன்பேர்க் ஒரு வித உளவுச்செய்தியை சோவியத் அதிகாரிகளுக்கு இரண்டாம் உலகப்போரின்போது அனுப்பியதில் சம்பந்தப்பட்டு இருந்ததாக வெளிவந்துள்ளன.
 <EOS>
[1, 1843, 69, 886, 1844, 1845, 3, 1846, 1847, 283, 551, 1848, 1849, 1850, 1851, 1852, 1853, 2] 
Length of sequence: 18


In [34]:
max_input_seq_length=20
max_output_seq_length=20


In [ ]:
# Pad sequences to a fixed length
# Pads or trims every English and Tamil sentence so they all have the same length, adding zeros at the end if needed.
input_sequences = pad_sequences(english_sequences, maxlen=max_input_seq_length, padding='post')
output_sequences = pad_sequences(tamil_sequences, maxlen=max_output_seq_length, padding='post')


In [44]:
print(input_sequences)
print(output_sequences)


[[   2 2036 2037 ...    0    0    0]
 [   9 2044 2045 ...  105  342    3]
 [   2    5 2047 ...    0    0    0]
 ...
 [6958 6959   68 ...   16 6963    3]
 [1449  207 6967 ... 6975 6976    3]
 [   2   45   85 ...    0    0    0]]
[[   1 1836   44 ...    0    0    0]
 [   1 1843   69 ...    2    0    0]
 [   1 1854 1855 ...    0    0    0]
 ...
 [9890 9891   86 ... 9899 9900    2]
 [9904 9905 9906 ... 1036 9919    2]
 [   1  195   25 ...    0    0    0]]


In [ ]:
# Prepare the decoder input and output sequences for teacher forcing
# We show the model the correct past words so it can learn the future ones faster and more accurately.

# 1. Creates an empty matrix with the same shape as your target sentences, to hold what the decoder will see as input
decoder_input_sequences = np.zeros_like(output_sequences)
# 2. Shifts every target sentence one step to the right so the decoder sees the previous word before predicting the next one.
decoder_input_sequences[:, 1:] = output_sequences[:, :-1]
# 3. Puts the start-of-sentence token at the beginning of every decoder input sequence.
decoder_input_sequences[:, 0] = tamil_tokenizer.word_index['<sos>']
# 4. Converts each target word index into a one-hot vector so the model learns to predict the next word correctly.
decoder_output_sequences = np.eye(tamil_vocab_size)[output_sequences]


In [ ]:
from gensim.models import Word2Vec
# import Word2Vec models we pretrained earlier
eng_model = Word2Vec.load('engmodel.bin')
tam_model = Word2Vec.load('tammodel.bin')


In [52]:
# Function that builds a matrix of pretrained word vectors aligned to the tokenizer’s vocabulary
def create_embedding_matrix(word2vec_model, tokenizer, vocab_size):
    # Creates an empty table where each row will store the embedding for one word in the vocabulary.
    embedding_matrix = np.zeros((vocab_size, word2vec_model.vector_size))
    # Loops through every word and its index from the tokenizer.
    for word, i in tokenizer.word_index.items():
        try:
            # Fetches the pretrained Word2Vec vector for that word.
            embedding_vector = word2vec_model.wv[word]
            # Places that vector into the row that matches the word’s index.
            embedding_matrix[i] = embedding_vector
        except KeyError:
            pass  # Words not found in the embedding index will be all zeros
    return embedding_matrix

eng_embedding_matrix = create_embedding_matrix(eng_model, english_tokenizer, english_vocab_size)
tam_embedding_matrix = create_embedding_matrix(tam_model, tamil_tokenizer, tamil_vocab_size)


In [53]:
eng_embedding_matrix.shape


(6979, 100)

In [ ]:
eng_embedding_matrix


array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.26573214,  0.61581212,  0.10475288, ..., -0.59400553,
        -0.18814734,  0.18171166],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.00245271, -0.00354816,  0.00910977, ..., -0.00299864,
         0.00671486,  0.00746955],
       [-0.00397189, -0.00433557,  0.00121044, ...,  0.00149533,
         0.00254785, -0.00227835]])

In [55]:
tam_embedding_matrix.shape


(9922, 100)

In [ ]:
tam_embedding_matrix


array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.00182868, -0.00939438,  0.00532814, ...,  0.00158221,
         0.00793402, -0.00312429],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]])

In [57]:
# function that builds a sequence-to-sequence translation model
def create_seq2seq_model(input_vocab_size, output_vocab_size, input_seq_length, output_seq_length, hidden_units, eng_embedding_matrix, tam_embedding_matrix):

    # Encoder
    # 1. Creates the input layer for the English sentence sequence.
    encoder_inputs = Input(shape=(input_seq_length,))
    # 2. Converts English word indices into pretrained vector representations.
    encoder_embedding = Embedding(input_vocab_size, hidden_units, weights=[eng_embedding_matrix], trainable=False)(encoder_inputs)
    # 3. Processes the English sentence and outputs the final memory states.
    encoder_lstm, encoder_state_h, encoder_state_c = LSTM(hidden_units, return_state=True)(encoder_embedding)

    # Decoder
    # 1. Creates the input layer for the Tamil target sentence.
    decoder_inputs = Input(shape=(output_seq_length,))
    # 2. Converts Tamil word indices into pretrained vector representations.
    decoder_embedding = Embedding(output_vocab_size, hidden_units, weights=[tam_embedding_matrix], trainable=False)(decoder_inputs)
    # 3. Defines the decoder LSTM that will generate the output sequence.
    decoder_lstm = LSTM(hidden_units, return_sequences=True, return_state=True)
    # 4. Starts the decoder using the encoder’s final memory states.
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=[encoder_state_h, encoder_state_c])
    # 5. Creates a layer that predicts the probability of every possible output word.
    decoder_dense = Dense(output_vocab_size, activation='softmax')
    # 6. Converts LSTM outputs into word probability distributions.
    decoder_outputs = decoder_dense(decoder_outputs)

    # Builds the full encoder–decoder model.
    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    return model


In [58]:
# Convert target_sequences to one-hot encoded format
target_sequences = tf.keras.utils.to_categorical(output_sequences, num_classes=tamil_vocab_size)


In [59]:
model = create_seq2seq_model(english_vocab_size, tamil_vocab_size, max_input_seq_length, max_output_seq_length, 100, eng_embedding_matrix, tam_embedding_matrix)


In [60]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [61]:
# Fit the model to the data
batch_size = 32
epochs = 500
model.fit([input_sequences, output_sequences], decoder_output_sequences, batch_size=batch_size, epochs=epochs, validation_split=0.2)


Epoch 1/500
25/25 [==============================] - 9s 186ms/step - loss: 8.9200 - accuracy: 0.2557 - val_loss: 8.3118 - val_accuracy: 0.2722
Epoch 2/500
25/25 [==============================] - 3s 131ms/step - loss: 7.4207 - accuracy: 0.2687 - val_loss: 7.4691 - val_accuracy: 0.2680
Epoch 3/500
25/25 [==============================] - 3s 136ms/step - loss: 6.5453 - accuracy: 0.2685 - val_loss: 7.4690 - val_accuracy: 0.2718
Epoch 4/500
25/25 [==============================] - 3s 125ms/step - loss: 6.3854 - accuracy: 0.2691 - val_loss: 7.5032 - val_accuracy: 0.2722
Epoch 5/500
25/25 [==============================] - 3s 123ms/step - loss: 6.3060 - accuracy: 0.2691 - val_loss: 7.5080 - val_accuracy: 0.2722
Epoch 6/500
25/25 [==============================] - 3s 125ms/step - loss: 6.2504 - accuracy: 0.2689 - val_loss: 7.5187 - val_accuracy: 0.2720
Epoch 7/500
25/25 [==============================] - 3s 133ms/step - loss: 6.1866 - accuracy: 0.2689 - val_loss: 7.5540 - val_accuracy: 0.2722

In [71]:
# Preprocessing the input
input_sentence = "<sos>Finally, the journalist fails to tell us who among the political leaders of the area, past and present, he counts among the paragons of morality<eos>"

# Convert the input sentence to sequence
input_sequence = english_tokenizer.texts_to_sequences([input_sentence])

# Pad the statement to the maximum input sequence length
input_sequence = pad_sequences(input_sequence, maxlen=max_input_seq_length, padding='post')

# Generate predictions
predictions = model.predict([input_sequence, np.zeros((1, max_output_seq_length))])

# Convert predictions to tokens
predicted_tokens = np.argmax(predictions, axis=-1)[0]

# Create index to word mapping for Tamil vocabulary
tamil_index_word = {i: w for w, i in tamil_tokenizer.word_index.items()}


# Convert tokens to text
decoded_sentence = []
for token in predicted_tokens:
    if token == 0:  # Assuming 0 is the padding token
        continue
    word = tamil_index_word.get(token)
    if word == '<eos>':
        break
    if word is not None:
        decoded_sentence.append(word)
    else:
        decoded_sentence.append('<unk>')

# Join the words to form the decoded statement
decoded_statement = ' '.join(decoded_sentence)

# Print the decoded statement
print(decoded_statement)


1/1 [==============================] - 0s 33ms/step
முந்நூறுபேரும் காரியம் வெளியில் அதில் வளையங்களையும் பண்ணி, விதத்தில், செனட் சந்தர்ப்பவாதத்துடனேயே பொறுமையிழந்துள்ள பயன்படுத்தி பக்கங்களிலே வைத்து,
 இயக்கத்தை தன்மையில் இயக்கத்தை நாம்


In [69]:
predictions


array([[[6.37270858e-08, 4.22042067e-04, 4.11001793e-07, ...,
         8.82705631e-10, 9.68128244e-10, 8.61826999e-10],
        [9.26749255e-09, 2.28647677e-05, 9.05751278e-08, ...,
         2.09273932e-09, 2.14114548e-09, 1.80166071e-09],
        [3.28261218e-09, 1.92540915e-06, 3.45268091e-07, ...,
         3.61798125e-09, 3.64079766e-09, 3.19371596e-09],
        ...,
        [1.07873944e-04, 3.00029293e-04, 3.18760782e-01, ...,
         2.06931458e-10, 2.11167417e-10, 1.78331613e-10],
        [1.46832748e-03, 1.87955040e-04, 6.97761536e-01, ...,
         1.49843637e-10, 1.59922631e-10, 1.36939446e-10],
        [1.85977221e-02, 6.75048432e-05, 8.44587505e-01, ...,
         7.77618456e-11, 8.61053867e-11, 7.62231875e-11]]], dtype=float32)

In [70]:
predicted_tokens


array([ 244, 2557,  345, 4366, 4367, 2402, 4472, 1381, 4473, 2403, 1024,
       2404, 1002,  372, 1399,  372, 5270,    2,    2,    2], dtype=int64)

In [65]:
input_sequence


array([[   6,  569,   35,   41,  172,    1,   67,  452,    4,    1, 1118,
         226,    5, 1119,   16, 2057,  172,    1, 2058,    4]])